# Apêndice — o mesmo pipeline em Spark (PySpark + MLflow)

> **Este notebook é um apêndice, não a arquitetura recomendada.** O book de wealth deste projeto tem ~1.200 relações — o problema menos "Big Data" possível, e `pandas` o resolve em segundos (ver `pipeline.py` e o notebook `01`). Ele existe para responder uma pergunta hipotética: *e se a carteira crescesse 100x?* Aí o processamento em nó único não escalaria, e a solução seria distribuir com Spark e rastrear o ciclo de vida do modelo com MLflow.

O conteúdo abaixo reimplementa a Direção A (churn do cliente) sobre o **mesmo dado sintético** (`output/data/base_clientes.csv`, gerado pelo `pipeline.py`), usando `pyspark.ml` no lugar do `scikit-learn` e MLflow para o tracking. As métricas aqui não são comparáveis 1:1 com as do pipeline principal: o split, o algoritmo (Random Forest do MLlib) e o pré-processamento são diferentes. O objetivo é mostrar o padrão de engenharia, não competir com a v2.


In [1]:
# Dependências deste apêndice (fora do requirements.txt do projeto, que é síncrono/pandas):
#   pip install pyspark mlflow
# Requer também um JDK (Java 17+) no PATH para o Spark subir a JVM.


### 2. Ligando o Motor do Cluster (Spark Session)
No mundo do Pandas, nós apenas importamos a biblioteca. No mundo do Big Data, nós precisamos iniciar uma **Spark Session**.

A Spark Session atua como o "Maestro" da orquestra: é ela quem recebe os seus comandos em Python e os traduz para a linguagem do motor do Spark (na JVM - Java Virtual Machine), distribuindo o processamento pela máquina.


In [2]:
from pyspark.sql import SparkSession

# Inicializando o Maestro (Cluster Local)
spark = SparkSession.builder \
    .appName("Churn_Financeiro_BigData") \
    .getOrCreate()

print("Motor do PySpark ligado com sucesso!")
print(f"Versão em execução: {spark.version}")


Motor do PySpark ligado com sucesso!
Versão em execução: 3.5.0


### 3. Ingestão de Dados (Zona Bronze)
Nossa primeira missão é carregar a base de clientes bruta.

A grande diferença estrutural aqui é: quando usamos `pd.read_csv()`, o Pandas lê o arquivo inteiro e o "enfia" goela abaixo na memória RAM.

Quando usamos `spark.read.csv()`, o Spark lê o arquivo, entende a estrutura dele e divide os dados em partições, sem estourar a memória.


In [3]:
# O dado é o mesmo book sintético gerado por `python pipeline.py` na raiz do projeto.
# header=True: a primeira linha é o cabeçalho; inferSchema=True: o Spark deduz os tipos.
df_clientes = spark.read.csv("../output/data/base_clientes.csv", header=True, inferSchema=True)

# No pandas usamos .head(), no Spark usamos .show()
print("Amostra dos Dados:")
df_clientes.show(5)

# No pandas usamos .info(), no Spark usamos .printSchema()
# O Schema é vital no Big Data para evitar travamentos de tipagem em tabelas gigantes
print("\nEsquema de Dados (Tipagem):")
df_clientes.printSchema()


Amostra dos Dados:


+----------+----------+-------------+------------+---------------+----------------+-----------+-----+
|cliente_id|  segmento|meses_cliente|qtd_produtos|retorno_12m_pct|freq_contato_mes|auc_milhoes|churn|
+----------+----------+-------------+------------+---------------+----------------+-----------+-----+
|  CLI00000|Alta Renda|          130|           7|           6.95|               3|      3.268|    1|
|  CLI00001|    Wealth|           12|           6|          19.12|               2|     47.588|    0|
|  CLI00002|   Private|           34|           5|          13.98|               3|     14.924|    0|
|  CLI00003|   Private|           38|           1|          17.37|               4|     28.297|    0|
|  CLI00004|Alta Renda|          139|           8|           7.91|               3|      4.652|    0|
+----------+----------+-------------+------------+---------------+----------------+-----------+-----+
only showing top 5 rows


Esquema de Dados (Tipagem):
root
 |-- cliente_id: string

### 4. Análise Exploratória Distribuída (EDA)
Para manipularmos as colunas no PySpark, nós importamos um pacote de "funções".

Ele nos permite fazer agrupamentos, cálculos de média e contagens de forma muito semelhante ao SQL, mas usando Python.

Vamos investigar como o `churn` (cancelamento) se comporta dependendo do `segmento` do cliente.


In [4]:
from pyspark.sql.functions import col, mean, count, round

print("Taxa de Churn por Segmento:")

# Agrupando por segmento, contando os clientes e tirando a média do churn (que é 0 ou 1)
df_analise = df_clientes.groupBy("segmento") \
    .agg(
        count("cliente_id").alias("total_clientes"),
        round(mean("churn") * 100, 2).alias("taxa_churn_pct")
    ) \
    .orderBy(col("taxa_churn_pct").desc())

df_analise.show()


Taxa de Churn por Segmento:


+-------------+--------------+--------------+
|     segmento|total_clientes|taxa_churn_pct|
+-------------+--------------+--------------+
|   Alta Renda|           539|         14.66|
|      Private|           447|         11.19|
|Family Office|            50|           6.0|
|       Wealth|           164|          4.88|
+-------------+--------------+--------------+



### 5. Feature Engineering: O Jeito PySpark (MLlib)
A preparação de dados para Machine Learning no Spark é muito diferente do `Scikit-Learn`.

Enquanto no Pandas nós fazemos *One-Hot Encoding* e passamos um DataFrame cheio de colunas para o modelo, o Spark exige que **todas as variáveis preditivas (features) sejam agrupadas em uma única coluna do tipo "Vetor"**.

Para isso, usaremos duas ferramentas clássicas do Big Data:

1. **StringIndexer:** Transforma textos (ex: 'Alta Renda', 'Wealth') em números (0.0, 1.0).

2. **VectorAssembler:** Pega todas as colunas numéricas e as "esmaga" em uma única coluna chamada `features`.


In [5]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# 1. Transformando o texto do 'segmento' em números
indexer = StringIndexer(inputCol="segmento", outputCol="segmento_index")
# O .fit() entende as categorias e o .transform() aplica a mudança
df_prep = indexer.fit(df_clientes).transform(df_clientes)

# 2. Definindo quais colunas vamos usar para prever o Churn
colunas_features = [
    "segmento_index",
    "meses_cliente",
    "qtd_produtos",
    "retorno_12m_pct",
    "freq_contato_mes",
    "auc_milhoes"
]

# 3. Juntando todas essas colunas em um único Vetor chamado 'features'
assembler = VectorAssembler(inputCols=colunas_features, outputCol="features")
df_final = assembler.transform(df_prep)

# Mostrando apenas a coluna target (churn) e o novo Vetor de features
print("Tabela preparada para o Robô de Machine Learning:")
df_final.select("cliente_id", "churn", "features").show(5, truncate=False)


Tabela preparada para o Robô de Machine Learning:


+----------+-----+-------------------------------+
|cliente_id|churn|features                       |
+----------+-----+-------------------------------+
|CLI00000  |1    |[0.0,130.0,7.0,6.95,3.0,3.268] |
|CLI00001  |0    |[2.0,12.0,6.0,19.12,2.0,47.588]|
|CLI00002  |0    |[1.0,34.0,5.0,13.98,3.0,14.924]|
|CLI00003  |0    |[1.0,38.0,1.0,17.37,4.0,28.297]|
|CLI00004  |0    |[0.0,139.0,8.0,7.91,3.0,4.652] |
+----------+-----+-------------------------------+
only showing top 5 rows



### 6. Rastreamento e Treinamento de Machine Learning (MLOps)
Em ambientes corporativos, você não treina um modelo apenas dando `.fit()`. Você precisa provar e rastrear qual foi o resultado.

Para isso utilizamos o **MLflow**. Ele "envolve" o nosso treinamento e anota (log) automaticamente quem treinou, quais foram os hiperparâmetros usados e as métricas resultantes, criando um histórico auditável do modelo.


In [6]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("churn_spark_apendice")
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1. Separando os dados em Treino (70%) e Teste (30%) no PySpark
train_data, test_data = df_final.randomSplit([0.7, 0.3], seed=42)

print(f"Registros de Treino: {train_data.count()}")
print(f"Registros de Teste: {test_data.count()}")

# 2. Iniciando a gravação do experimento no MLflow
# O MLflow começa a gravar tudo que acontecer dentro deste bloco 'with'
with mlflow.start_run(run_name="RandomForest_Churn_V1"):

    # Configurando o Algoritmo (PySpark MLlib)
    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="churn",
        numTrees=100,
        maxDepth=5
    )

    # Treinando o Robô
    print("\nTreinando o modelo de Random Forest em Cluster (PySpark)...")
    rf_model = rf.fit(train_data)

    # 3. Fazendo Previsões na base de Teste
    previsoes = rf_model.transform(test_data)

    # 4. Avaliando a qualidade do modelo (Acurácia / F1-Score)
    evaluator = MulticlassClassificationEvaluator(
        labelCol="churn",
        predictionCol="prediction",
        metricName="f1"
    )

    f1_score = evaluator.evaluate(previsoes)

    # 5. Salvando o registro no MLflow
    # Aqui o MLOps acontece: guardamos os hiperparâmetros e o resultado no cofre!
    mlflow.log_param("algoritmo", "Random Forest")
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_metric("f1_score", f1_score)

    print(f"\n[SUCESSO] Modelo Treinado! F1-Score do Teste: {f1_score:.4f}")
    print("Os dados do experimento foram salvos pelo MLflow de forma segura.")


2026/09/10 13:51:08 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\Luiz Maibashi\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


2026/09/10 13:51:09 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/09/10 13:51:09 INFO mlflow.store.db.utils: Updating database tables


2026/09/10 13:51:11 INFO mlflow.tracking.fluent: Experiment with name 'churn_spark_apendice' does not exist. Creating a new experiment.


Registros de Treino: 887


Registros de Teste: 313

Treinando o modelo de Random Forest em Cluster (PySpark)...



[SUCESSO] Modelo Treinado! F1-Score do Teste: 0.8264
Os dados do experimento foram salvos pelo MLflow de forma segura.


### 7. Inspecionando o Resultado do Modelo
Vamos ver como a tabela sai do outro lado após passar pelo modelo.
O PySpark adiciona novas colunas no final com as probabilidades e a previsão final.


In [7]:
# Mostrando o cruzamento: o que era realidade (churn), as probabilidades do modelo, e o que o modelo previu
previsoes.select("cliente_id", "churn", "probability", "prediction").show(10, truncate=False)


+----------+-----+----------------------------------------+----------+
|cliente_id|churn|probability                             |prediction|
+----------+-----+----------------------------------------+----------+
|CLI00002  |0    |[0.912240618319231,0.08775938168076908] |0.0       |
|CLI00006  |1    |[0.8882915356984575,0.11170846430154248]|0.0       |
|CLI00008  |0    |[0.8723772600055023,0.12762273999449764]|0.0       |
|CLI00009  |0    |[0.8764734446084138,0.12352655539158615]|0.0       |
|CLI00013  |0    |[0.8232675137677079,0.1767324862322922] |0.0       |
|CLI00014  |0    |[0.8910081778077164,0.10899182219228364]|0.0       |
|CLI00015  |0    |[0.7409823189892228,0.25901768101077716]|0.0       |
|CLI00019  |1    |[0.747965479277865,0.252034520722135]   |0.0       |
|CLI00021  |0    |[0.8784466625106124,0.12155333748938751]|0.0       |
|CLI00023  |0    |[0.8695598826581487,0.13044011734185137]|0.0       |
+----------+-----+----------------------------------------+----------+
only s

O modelo desbalanceado prevê `0.0` para todos os clientes da amostra, inclusive para o `CLI00006`, que de fato deu churn. Com ~12% de eventos positivos, a Random Forest minimiza o erro global ignorando a classe minoritária: acerta a maioria e nunca "arrisca" um alarme.

Isso é o efeito clássico de **desbalanceamento de classes** — o F1 ponderado fica alto (~0,83) escondendo recall ~0 na classe que interessa.


In [8]:
# Contando a proporção de Churn
df_clientes.groupBy("churn").count().show()


+-----+-----+
|churn|count|
+-----+-----+
|    1|  140|
|    0| 1060|
+-----+-----+



### 8. Resolvendo o Desbalanceamento (Class Weights)

A base tem 140 churns em 1.200 relações (~12%). Vamos compensar isso com um **peso por classe**: quem tem menos exemplos ganha peso maior na função de perda, forçando a árvore a prestar atenção nos churns. O peso é calculado da proporção, não chutado.


In [9]:
from pyspark.sql.functions import when

# 1. Calculando os pesos automaticamente (sem chutar números)
total_clientes = df_final.count()
qtd_churn = df_final.filter(col("churn") == 1).count()
qtd_fieis = total_clientes - qtd_churn

# A matemática do peso: quem tem menos, ganha peso maior
peso_churn = total_clientes / (2 * qtd_churn)
peso_fiel = total_clientes / (2 * qtd_fieis)

print(f"Peso aplicado para clientes Fiéis (0): {peso_fiel:.2f}")
print(f"Peso aplicado para clientes de Churn (1): {peso_churn:.2f}")

# 2. Criando uma nova coluna na base com o peso de cada cliente
df_balanceado = df_final.withColumn(
    "peso_classe",
    when(col("churn") == 1, peso_churn).otherwise(peso_fiel)
)

# 3. Refazendo a separação Treino/Teste
train_data_bal, test_data_bal = df_balanceado.randomSplit([0.7, 0.3], seed=42)


Peso aplicado para clientes Fiéis (0): 0.57
Peso aplicado para clientes de Churn (1): 4.29


### 9. Treinando o Modelo V2 (Balanceado) no MLflow
Agora vamos rodar o modelo novamente, mas desta vez passaremos o parâmetro `weightCol` para o Random Forest, dizendo a ele para respeitar a punição imposta pela nossa nova coluna.


In [10]:
# Iniciando uma NOVA gravação no MLflow para a Versão 2
with mlflow.start_run(run_name="RandomForest_Churn_V2_Balanceado"):

    # Repare no novo parâmetro: weightCol="peso_classe"
    rf_v2 = RandomForestClassifier(
        featuresCol="features",
        labelCol="churn",
        weightCol="peso_classe", # A MÁGICA ACONTECE AQUI
        numTrees=100,
        maxDepth=5
    )

    print("Treinando o modelo V2 Balanceado...")
    rf_model_v2 = rf_v2.fit(train_data_bal)

    previsoes_v2 = rf_model_v2.transform(test_data_bal)

    # Avaliando
    f1_score_v2 = evaluator.evaluate(previsoes_v2)

    # Salvando no MLflow
    mlflow.log_param("algoritmo", "Random Forest")
    mlflow.log_param("balanceamento", "Class Weights")
    mlflow.log_metric("f1_score", f1_score_v2)

    print(f"\n[SUCESSO] Modelo V2 Treinado! Novo F1-Score: {f1_score_v2:.4f}")

# Mostrando o resultado lado a lado novamente
print("\nNovas Previsões (Observe se agora aparecem previsões '1.0'):")
previsoes_v2.select("cliente_id", "churn", "probability", "prediction").show(10, truncate=False)


Treinando o modelo V2 Balanceado...



[SUCESSO] Modelo V2 Treinado! Novo F1-Score: 0.7811

Novas Previsões (Observe se agora aparecem previsões '1.0'):


+----------+-----+----------------------------------------+----------+
|cliente_id|churn|probability                             |prediction|
+----------+-----+----------------------------------------+----------+
|CLI00002  |0    |[0.6297689738757071,0.3702310261242929] |0.0       |
|CLI00006  |1    |[0.5163086834367296,0.4836913165632703] |0.0       |
|CLI00008  |0    |[0.5820398840296886,0.4179601159703113] |0.0       |
|CLI00009  |0    |[0.5767925437106015,0.4232074562893984] |0.0       |
|CLI00013  |0    |[0.532433875736743,0.467566124263257]   |0.0       |
|CLI00014  |0    |[0.5454155660051329,0.4545844339948672] |0.0       |
|CLI00015  |0    |[0.5866044395543479,0.4133955604456521] |0.0       |
|CLI00019  |1    |[0.48702596388709973,0.5129740361129003]|1.0       |
|CLI00021  |0    |[0.533716329249763,0.466283670750237]   |0.0       |
|CLI00023  |0    |[0.47742627345178384,0.5225737265482162]|1.0       |
+----------+-----+----------------------------------------+----------+
only s

In [11]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def recall_churn(previsoes_df):
    ev = MulticlassClassificationEvaluator(
        labelCol="churn", predictionCol="prediction",
        metricName="recallByLabel", metricLabel=1.0,
    )
    return ev.evaluate(previsoes_df)

r1 = recall_churn(previsoes)      # modelo desbalanceado
r2 = recall_churn(previsoes_v2)   # modelo com class weights
print(f"Recall da classe churn — desbalanceado: {r1:.3f}")
print(f"Recall da classe churn — balanceado:    {r2:.3f}")
print(f"F1 ponderado — desbalanceado: {f1_score:.3f} | balanceado: {f1_score_v2:.3f}")


Recall da classe churn — desbalanceado: 0.000
Recall da classe churn — balanceado:    0.216
F1 ponderado — desbalanceado: 0.826 | balanceado: 0.781


Os pesos de classe **aumentam a probabilidade atribuída aos churns** (compare a coluna `probability` antes e depois) e elevam o recall da classe minoritária. Em troca, o F1 ponderado cai: o modelo passa a errar mais falsos positivos para não perder churn.

Com uma floresta rasa (`maxDepth=5`) sobre ~12% de prevalência, a virada é parcial, não total — muitas relações de churn ainda ficam abaixo do corte 0,5. O ajuste completo exigiria calibrar o **threshold de decisão** pela curva de custo (é o que o `pipeline.py` da v2 faz em `calibrate_thresholds_v2`), não só reponderar o treino. Reponderação e threshold são passos complementares, não intercambiáveis.


**Por que priorizar recall na classe churn:** um falso negativo (deixar uma relação de wealth deteriorar sem ver o sinal) custa a receita recorrente daquele AuC; um falso positivo custa uma ligação do assessor ou uma oferta. A assimetria justifica aceitar mais falsos positivos para não perder churn — no dado sintético, é uma ilustração do trade-off, não um valor de negócio medido.
